# EBSM
- author: Muxin, Beatrice
- date: 06/07/2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs

## 1. Box model for prognostic energy (i.e., Temperature) in surface/deep ocean

In [ ]:
# define the constant
c_star = 10.8e9               # J K-1 m-2
delta_surfocean = 0.015
lambda_ = 1.77                # W m-1 K-2
lambda_star = 0.75           # W m-1 K-2
beta = 5.77                   # W m-2
eta_H = 1                     # W m-2 K-1

### 1.1 surface ocean temperature prognostic

#### Forward functions

In [ ]:
def euler_forward_func(dt, F_n, dF_n):
    """
    General Euler forward function,
    Will be used in the first time step
    """
    
    F_np1 = F_n + dt * dF_n
    
    return F_np1

def leapfrog_forward_func(dt, F_nm1, dF_n):
    """
    General Leapfrog forward function,
    Will be used from the 2nd timestep onward
    """
    
    F_np1 = F_nm1 + 2 * dt * dF_n
    
    return F_np1

#### Tendency functions

In [ ]:
# surface ocean
def T_s_tendency(T_s_n, T_D_n, C_a_n, dS0_n, C_a_0, alpha_0, S0_change=False):
    """
    the time derivative of surface ocean temperature 
    at current timestep

    Input:
    ------
    - T_s_n: surface ocean temperature at current timestep
    - T_D_n: deep ocean temperature at current timestep
    - C_a_n: atmospheric CO2 concentration at current timestep
    - C_a_0: pre-industrial atmospheric CO2 concentration
    - dS0_n: change in solar constant at current timestep
    - constant:
        - alpha_0: pre-industrial albedo
        - eta_H: heat exchange coefficient
        - S0: solar constant
        - lambda_: Radiativeresponse to surface temp anomaly 
            (climate sensitivity parameter)
        - lambda_star: Radiative response surface-deep ocean disequilibrium
        - beta: Radiative response to CO2 concentration
    """
    # define the constants:
    c_star = 10.8e9               # J K-1 m-2
    delta_surfocean = 0.015
    lambda_ = 1.77                # W m-1 K-2
    lambda_star = 0.75           # W m-1 K-2
    beta = 5.77                   # W m-2
    eta_H = 1                     # W m-2 K-1

    # terms in tendency
    c_s = delta_surfocean * c_star          # heat capacity of surface ocean
    A = lambda_ * T_s_n                    # radiative response to surface temperature anomaly (linear)
    B = beta * np.ln(C_a_n / C_a_0 + 1)    # radiative response to CO2 concentration
    C = -eta_H * (T_s_n - T_D_n)           # energy exchange between surface and deep ocean
    D = -lambda_star * (T_s_n - T_D_n)     # alpha derivative of radiative response to surface-deep ocean disequilibrium
    
    if S0_change:
        E = (1/4) * dS0_n * (1 - alpha_0)  # change in solar constant
        dT_s_n = (1/c_s) * (A + B + C + D + E)
    else:
        dT_s_n = (1/c_s) * (A + B + C + D)

    return dT_s_n


# deep ocean
def T_D_tendency(T_s_n, T_D_n):
    """
    the time derivative of deep ocean temperature 
    at current timestep

    Input:
    ------
    - T_s_n: surface ocean temperature at current timestep
    - T_D_n: deep ocean temperature at current timestep
    - constant:
        - eta_H: heat exchange coefficient
    """
    # define the constants:
    eta_H = 1                     # W m-2 K-1
    c_star = 10.8e9               # J K-1 m-2
    delta_surfocean = 0.015

    # terms in tendency
    c_d = (1 - delta_surfocean) * c_star          # heat capacity of deep ocean
    dT_D_n = eta_H * ((T_s_n - T_D_n) / c_d)
    return dT_D_n

SyntaxError: invalid syntax (849499257.py, line 31)